# 🧹 Data Cleaning — IoT-Integrated Predictive Maintenance Dataset

## Objetivo
Preparar o dataset de manutenção preditiva baseado em sensores IoT para modelagem, a partir dos achados da análise exploratória (`01_eda.ipynb`):
- Tratar valores ausentes identificados na EDA
- Remover ou tratar duplicatas
- Corrigir tipos de dados das colunas
- Tratar outliers das variáveis de sensores
- Padronizar nomes de colunas e variáveis categóricas
- Exportar um dataset limpo, pronto para a etapa de modelagem

## Dataset
- **Fonte:** https://www.kaggle.com/datasets/ziya07/iot-integrated-predictive-maintenance-dataset/data
- **Entrada:** dataset bruto usado em `01_eda.ipynb`
- **Saída:** dataset limpo salvo em `data/processed/`

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

SENSOR_COLS = ['vibration', 'acoustic', 'temperature', 'current', 'IMF_1', 'IMF_2', 'IMF_3']
TARGET_COL = 'label'
ID_COL = 'machine_id'
TIME_COL = 'timestamp'

## 1. Carregamento dos dados

In [8]:


# Caminho do dataset bruto (ajustar conforme necessário)
RAW_PATH = "data/raw/iot_predictive_maintenance.csv"

df = pd.read_csv('../data/predictive_maintenance_dataset.csv')
df.shape


(1800, 10)

## 2. Inspeção inicial

In [9]:
df.head()

,timestamp,machine_id,vibration,acoustic,temperature,current,IMF_1,IMF_2,IMF_3,label
0,2024-07-01 08:00:00,M01,0.822,0.645,66.85,13.04,0.196,0.033,0.000,0
1,2024-07-01 08:01:00,M01,1.398,0.834,76.20,15.08,0.345,0.132,0.001,1
2,2024-07-01 08:02:00,M01,0.856,0.590,67.03,12.30,0.187,0.017,0.002,0
3,2024-07-01 08:03:00,M01,0.793,0.544,65.04,11.69,0.196,-0.060,0.003,0
4,2024-07-01 08:04:00,M01,1.279,0.721,78.19,14.84,0.330,-0.115,0.004,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   timestamp    1800 non-null   object 
 1   machine_id   1800 non-null   object 
 2   vibration    1800 non-null   float64
 3   acoustic     1800 non-null   float64
 4   temperature  1800 non-null   float64
 5   current      1800 non-null   float64
 6   IMF_1        1800 non-null   float64
 7   IMF_2        1800 non-null   float64
 8   IMF_3        1800 non-null   float64
 9   label        1800 non-null   int64  
dtypes: float64(7), int64(1), object(2)
memory usage: 140.8+ KB


In [6]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,1800,600,2024-07-01 08:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
machine_id,1800,3,M01,600,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vibration,1800.0,NaN,NaN,NaN,0.843295,0.13669,0.645,0.769,0.805,0.851,1.405
acoustic,1800.0,NaN,NaN,NaN,0.633898,0.107989,0.43,0.572,0.609,0.65125,1.083
temperature,1800.0,NaN,NaN,NaN,66.355722,4.448129,58.36,63.73,65.355,66.99,85.16
current,1800.0,NaN,NaN,NaN,12.3286,1.09451,10.29,11.7,12.08,12.48,16.94
IMF_1,1800.0,NaN,NaN,NaN,0.168738,0.056533,0.073,0.123,0.166,0.207,0.393
IMF_2,1800.0,NaN,NaN,NaN,0.000072,0.073104,-0.18,-0.058,-0.001,0.059,0.187
IMF_3,1800.0,NaN,NaN,NaN,0.000673,0.036034,-0.05,-0.037,0.0035,0.037,0.05
label,1800.0,NaN,NaN,NaN,0.112222,0.315727,0.0,0.0,0.0,0.0,1.0


## 3. Padronização de nomes de colunas

Garantir nomes consistentes (sem espaços/maiúsculas inconsistentes), mantendo `IMF_1`, `IMF_2`, `IMF_3` no padrão já usado na EDA.

In [10]:
df.columns = df.columns.str.strip()
df.columns.tolist()

['timestamp',
 'machine_id',
 'vibration',
 'acoustic',
 'temperature',
 'current',
 'IMF_1',
 'IMF_2',
 'IMF_3',
 'label']

## 4. Correção de tipos de dados

Converter `timestamp` para `datetime` e `machine_id` para `category`. 

In [12]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce')
df[ID_COL] = df[ID_COL].astype('category')
df[TARGET_COL] = df[TARGET_COL].astype(int)

df.dtypes

timestamp      datetime64[ns]
machine_id           category
vibration             float64
acoustic              float64
temperature           float64
current               float64
IMF_1                 float64
IMF_2                 float64
IMF_3                 float64
label                   int64
dtype: object

## 5. Valores ausentes

Na EDA, `count` = 1800 em todas as colunas, indicando ausência de nulos. Ainda assim, verificamos aqui para garantir — e após a conversão de `timestamp` (que pode gerar `NaT` em caso de formato inválido).

In [13]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'ausentes': missing, 'percentual_%': missing_pct})
missing_df[missing_df['ausentes'] > 0].sort_values('percentual_%', ascending=False)

,ausentes,percentual_%


In [14]:
# Caso existam ausentes nos sensores, imputar por mediana DENTRO de cada máquina
# (sensores de máquinas diferentes têm baselines distintos)
for col in SENSOR_COLS:
    if df[col].isnull().sum() > 0:
        df[col] = df.groupby(ID_COL)[col].transform(lambda s: s.fillna(s.median()))

# Linhas com timestamp inválido (NaT) — normalmente é mais seguro remover
n_nat = df[TIME_COL].isnull().sum()
if n_nat > 0:
    df = df.dropna(subset=[TIME_COL])

df.isnull().sum().sum()

np.int64(0)